In [3]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [7]:
import sys
sys.path.append('../../')
from model import FinData
from model import train_valid_test_split
from model import CatboostFinModel

import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import pandas as pd
from catboost import CatBoostClassifier

Выгружаем лонг и шорт данные 

In [32]:
# Лонг

import pandas as pd
import datetime as dt
from catboost import CatBoostClassifier

args1 = {
    "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : 7,
        "l2_leaf_reg" : 149,  
}

companies_names = ['MMK']
dfs = ["../../datasets/" + stock + '_1_min.csv' for stock in companies_names]

cutoff_time = dt.datetime(2024, 2, 1)
start_time = dt.datetime(2024, 5, 1)
end_time = dt.datetime(2024, 12, 31)

for i in range(len(dfs)):
    res = pd.DataFrame()
    findata = FinData(dfs[i])
    findata.restrict_time_down(cutoff_time)
    findata.insert_all()

    cat_feats = findata.cat_features
    num_feats = findata.numeric_features
    target = 'direction_binary_0'
    
    curr_time = start_time
    company_name = companies_names[i]

    while curr_time < end_time:
        data = findata.df
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        train_sd, val_sd, test_sd = train_df["utc"].iloc[0], val_df["utc"].iloc[0], test_df["utc"].iloc[0]
        train_ed, val_ed, test_ed = train_df["utc"].iloc[-1], val_df["utc"].iloc[-1], test_df["utc"].iloc[-1]
        print(f"Начало тренировочного периода: {train_sd}. Конец тренировочного периода: {train_ed} \n \
                    Начало валидационного периода: {val_sd}. Конец валидационного периода: {val_ed} \n \
                    Начало тестового периода: {test_sd}. Конец тестового периода: {test_ed} \n ")
        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        model = CatBoostClassifier(
            **args1
        )

        # Передаем validation_set через eval_set
        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
        )

        # Предсказания вероятностей класса 1
        proba = model.predict_proba(X_test)[:, 1]

        # Формирование блока с результатами
        temp_df = pd.DataFrame({
            'utc': test_df['utc'].values,
            'open': test_df['open'].values,
            'close': test_df['close'].values,
            'predicted_proba': proba
        })

        # Добавляем в общий DataFrame
        res = pd.concat([res, temp_df], ignore_index=True)

        # Сдвигаем окно
        curr_time += dt.timedelta(days=5)

    # Сохраняем результат по компании
    res.to_csv(f'{company_name}_long_valid.csv', index=False)


Начало тренировочного периода: 2024-03-27 07:00:00. Конец тренировочного периода: 2024-04-25 20:49:00 
                     Начало валидационного периода: 2024-04-26 07:00:00. Конец валидационного периода: 2024-04-30 20:49:00 
                     Начало тестового периода: 2024-05-01 07:01:00. Конец тестового периода: 2024-05-05 20:49:00 
 
Начало тренировочного периода: 2024-04-01 07:00:00. Конец тренировочного периода: 2024-04-30 20:49:00 
                     Начало валидационного периода: 2024-05-01 07:01:00. Конец валидационного периода: 2024-05-05 20:49:00 
                     Начало тестового периода: 2024-05-06 07:00:00. Конец тестового периода: 2024-05-10 20:49:00 
 
Начало тренировочного периода: 2024-04-06 07:01:00. Конец тренировочного периода: 2024-05-05 20:49:00 
                     Начало валидационного периода: 2024-05-06 07:00:00. Конец валидационного периода: 2024-05-10 20:49:00 
                     Начало тестового периода: 2024-05-11 07:01:00. Конец тестового пер

In [33]:
# Шорт

import pandas as pd
import datetime as dt

args1 = {
    "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : 7,
        "l2_leaf_reg" : 40,  
}

companies_names = ['MMK']
dfs = ["../../datasets/" + stock + '_1_min.csv' for stock in companies_names]

cutoff_time = dt.datetime(2024, 2, 1)
start_time = dt.datetime(2024, 5, 1)
end_time = dt.datetime(2024, 12, 31)

for i in range(len(dfs)):
    res = pd.DataFrame()
    findata = FinData(dfs[i])
    findata.restrict_time_down(cutoff_time)
    findata.insert_all()

    cat_feats = findata.cat_features
    num_feats = findata.numeric_features
    target = 'direction_binary_1'
    
    curr_time = start_time
    company_name = companies_names[i]

    while curr_time < end_time:
        data = findata.df
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        train_sd, val_sd, test_sd = train_df["utc"].iloc[0], val_df["utc"].iloc[0], test_df["utc"].iloc[0]
        train_ed, val_ed, test_ed = train_df["utc"].iloc[-1], val_df["utc"].iloc[-1], test_df["utc"].iloc[-1]
        print(f"Начало тренировочного периода: {train_sd}. Конец тренировочного периода: {train_ed} \n \
                    Начало валидационного периода: {val_sd}. Конец валидационного периода: {val_ed} \n \
                    Начало тестового периода: {test_sd}. Конец тестового периода: {test_ed} \n ")
        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        
        model = CatBoostClassifier(
            **args1
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
        )
        # Предсказания вероятностей класса 1
        proba = model.predict_proba(X_test)[:, 1]

        # Формирование блока с результатами
        temp_df = pd.DataFrame({
            'utc': test_df['utc'].values,
            'open': test_df['open'].values,
            'close': test_df['close'].values,
            'predicted_proba': proba
        })

        # Добавляем в общий DataFrame
        res = pd.concat([res, temp_df], ignore_index=True)

        # Сдвигаем окно
        curr_time += dt.timedelta(days=5)

    # Сохраняем результат по компании
    res.to_csv(f'{company_name}_short_valid.csv', index=False)


Начало тренировочного периода: 2024-03-27 07:00:00. Конец тренировочного периода: 2024-04-25 20:49:00 
                     Начало валидационного периода: 2024-04-26 07:00:00. Конец валидационного периода: 2024-04-30 20:49:00 
                     Начало тестового периода: 2024-05-01 07:01:00. Конец тестового периода: 2024-05-05 20:49:00 
 
Начало тренировочного периода: 2024-04-01 07:00:00. Конец тренировочного периода: 2024-04-30 20:49:00 
                     Начало валидационного периода: 2024-05-01 07:01:00. Конец валидационного периода: 2024-05-05 20:49:00 
                     Начало тестового периода: 2024-05-06 07:00:00. Конец тестового периода: 2024-05-10 20:49:00 
 
Начало тренировочного периода: 2024-04-06 07:01:00. Конец тренировочного периода: 2024-05-05 20:49:00 
                     Начало валидационного периода: 2024-05-06 07:00:00. Конец валидационного периода: 2024-05-10 20:49:00 
                     Начало тестового периода: 2024-05-11 07:01:00. Конец тестового пер

In [34]:
import datetime as dt
from sklearn.metrics import accuracy_score
import optuna
import numpy as np

# ваши подготовка дата/фрейма здесь не меняется
company_name = 'Whoosh'
df_path = '../../datasets/' + company_name + '_1_min.csv'

start_time = dt.datetime(2024, 2, 1)
end_time = dt.datetime(2024, 4, 30)
cutoff_time = start_time - dt.timedelta(days=90)

findata = FinData(df_path)
findata.restrict_time_down(cutoff_time)
findata.restrict_time_up(end_time + dt.timedelta(days=2))
findata.insert_all()

cat_feats = findata.cat_features
num_feats = findata.numeric_features
target = 'direction_binary_0'

data = findata.df

def objective(trial):

    params={
        "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : trial.suggest_int("depth", 2, 7),
        "l2_leaf_reg" : trial.suggest_int("l2_leaf_reg", 3, 200),    
    }

    curr_time = start_time
    f_betas = []

    while curr_time < end_time:
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        if test_df.empty:
            break

        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        model = CatBoostClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba > 0.5).astype(int)
        f_betas.append(accuracy_score(y_test, preds))
        curr_time += dt.timedelta(days=5)

    final = np.mean(f_betas)
    return final

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=7)

print("Лучшие параметры:")
print(study.best_params)
print("Лучшее значение Accuracy:")
print(study.best_value)


[I 2025-07-08 08:43:56,921] A new study created in memory with name: no-name-4523d74c-4cb3-4698-976e-2bda1900d1a6
[W 2025-07-08 08:46:36,741] Trial 0 failed with parameters: {'depth': 6, 'l2_leaf_reg': 171} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/Users/naburkova/repos/prices-predictions-1/venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/wt/lsf6j96n0xnc67mjwrs1l5p00000gn/T/ipykernel_49928/2843545266.py", line 56, in objective
    model.fit(
  File "/Users/naburkova/repos/prices-predictions-1/venv/lib/python3.11/site-packages/catboost/core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "/Users/naburkova/repos/prices-predictions-1/venv/lib/python3.11/site-packages/catboost/core.p

KeyboardInterrupt: 

In [31]:
import datetime as dt
import optuna
import numpy as np

# ваши подготовка дата/фрейма здесь не меняется
# company_name = 'Gazprom'
# df_path = '../../datasets/' + company_name + '_1_min.csv'

start_time = dt.datetime(2024, 2, 1)
end_time = dt.datetime(2024, 4, 30)
cutoff_time = start_time - dt.timedelta(days=90)

findata = FinData(df_path)
findata.restrict_time_down(cutoff_time)
findata.restrict_time_up(end_time + dt.timedelta(days=2))
findata.insert_all()

cat_feats = findata.cat_features
num_feats = findata.numeric_features
target = 'direction_binary_1'

data = findata.df

def objective(trial):

    params={
        "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : trial.suggest_int("depth", 2, 7),
        "l2_leaf_reg" : trial.suggest_int("l2_leaf_reg", 3, 200),    
    }

    curr_time = start_time
    f_betas = []

    while curr_time < end_time:
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        if test_df.empty:
            break

        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        model = CatBoostClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba > 0.5).astype(int)
        f_betas.append(accuracy_score(y_test, preds))
        curr_time += dt.timedelta(days=5)

    final = np.mean(f_betas)
    return final

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=7)

print("Лучшие параметры:")
print(study.best_params)
print("Лучшее значение Accuracy:")
print(study.best_value)


[I 2025-07-07 19:28:53,832] A new study created in memory with name: no-name-33cf5d7d-39be-404e-97ab-a6d88a0a94de
[I 2025-07-07 19:33:22,519] Trial 0 finished with value: 0.6241223797287614 and parameters: {'depth': 7, 'l2_leaf_reg': 50}. Best is trial 0 with value: 0.6241223797287614.
[I 2025-07-07 19:37:14,214] Trial 1 finished with value: 0.6239094127795177 and parameters: {'depth': 6, 'l2_leaf_reg': 187}. Best is trial 0 with value: 0.6241223797287614.
[I 2025-07-07 19:40:18,366] Trial 2 finished with value: 0.6239668688426249 and parameters: {'depth': 5, 'l2_leaf_reg': 151}. Best is trial 0 with value: 0.6241223797287614.
[I 2025-07-07 19:43:39,356] Trial 3 finished with value: 0.623749722573195 and parameters: {'depth': 6, 'l2_leaf_reg': 60}. Best is trial 0 with value: 0.6241223797287614.
[I 2025-07-07 19:46:27,004] Trial 4 finished with value: 0.6237822121893878 and parameters: {'depth': 5, 'l2_leaf_reg': 72}. Best is trial 0 with value: 0.6241223797287614.
[I 2025-07-07 19:50:

Лучшие параметры:
{'depth': 7, 'l2_leaf_reg': 40}
Лучшее значение Accuracy:
0.6242334653383255
